Big Data 2026/2027 ősz | Data Science Beadandó | Dékán Máté (GQHTIH)

Importok és Beállítások

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import warnings

warnings.filterwarnings('ignore')
print("imported modules")

imported modules


Adatbeolvasás és Duplikátumok Kezelése

In [3]:
train_df = pd.read_csv('data/input.csv')
val_df = pd.read_csv('data/validation.csv')

print(f"Tanító adat eredeti alakja: {train_df.shape}")
print(f"Validációs adat alakja: {val_df.shape}\n")

dup_count = train_df.duplicated().sum()
print(f"Teljes soros duplikátumok száma: {dup_count}")

# Teljes soros duplikátumok eltávolítása drop_duplicates() segítségével
if dup_count > 0:
    train_df = train_df.drop_duplicates().reset_index(drop=True)
    print(f"Duplikátumok törölve! Új alak: {train_df.shape}")

print("\nCélváltozó ('class') eloszlása:")
print(train_df['class'].value_counts(dropna=False))

Tanító adat eredeti alakja: (39084, 9)
Validációs adat alakja: (9771, 8)

Teljes soros duplikátumok száma: 0

Célváltozó ('class') eloszlása:
class
p    21716
e    17368
Name: count, dtype: int64


Előfeldolgozás: dátumból hónap, hiányzó értékek kezelése

In [ ]:
NUM_COLS = ['cap-diameter', 'stem-height', 'stem-width']
CAT_COLS = ['cap-shape', 'cap-color', 'has-ring', 'ring-type', 'month']

def preprocess(df, num_medians):
    df = df.copy()
    # A dátumból csak a hónapot tartjuk meg (szezonalitás), kategóriaként kezelve
    df['month'] = pd.to_datetime(df['date'], format='%Y.%m.%d').dt.month.astype(str)
    df = df.drop(columns=['date'])
    # Hiányzó kategória: saját 'missing' értéket kap
    df['ring-type'] = df['ring-type'].fillna('missing')
    # Hiányzó numerikus érték: tanító halmaz mediánjával pótoljuk
    df[NUM_COLS] = df[NUM_COLS].fillna(num_medians)
    return df

medians = train_df[NUM_COLS].median()
X = preprocess(train_df.drop(columns=['class']), medians)
y = (train_df['class'] == 'p').astype(int)  # 1 = mérgező
X_val = preprocess(val_df, medians)

print("Hiányzó értékek előfeldolgozás után:", X.isna().sum().sum(), "/", X_val.isna().sum().sum())
X.head()

Modell: Pipeline (skálázás + one-hot) + logisztikus regresszió, 5-szörös rétegzett keresztvalidáció

In [ ]:
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), NUM_COLS),
    ('cat', OneHotEncoder(handle_unknown='ignore'), CAT_COLS),
])
model = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression(max_iter=2000, C=1.0)),
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = {'accuracy': [], 'precision': [], 'recall': [], 'f1': []}
for fold, (tr, te) in enumerate(skf.split(X, y), 1):
    model.fit(X.iloc[tr], y.iloc[tr])
    pred = model.predict(X.iloc[te])
    scores['accuracy'].append(accuracy_score(y.iloc[te], pred))
    scores['precision'].append(precision_score(y.iloc[te], pred))
    scores['recall'].append(recall_score(y.iloc[te], pred))
    scores['f1'].append(f1_score(y.iloc[te], pred))
    print(f"Fold {fold}: acc={scores['accuracy'][-1]:.4f}  f1={scores['f1'][-1]:.4f}")

print()
for k, v in scores.items():
    print(f"{k:>9}: {np.mean(v):.4f} ± {np.std(v):.4f}")

Végső modell tanítása a teljes tanító halmazon, predikció a validációs halmazra

In [ ]:
model.fit(X, y)
print(classification_report(y, model.predict(X), target_names=['e', 'p']))

val_pred = np.where(model.predict(X_val) == 1, 'p', 'e')
with open('pred.txt', 'w') as f:
    f.write('
'.join(val_pred) + '
')

print(f"pred.txt mentve: {len(val_pred)} sor")
print(pd.Series(val_pred).value_counts())